In [26]:
# 0. Cargamos datos

import pandas as pd
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest

df_demo    = pd.read_csv('Datasets_limpios/clean_client_profiles.csv')
df_web     = pd.read_csv('Datasets_limpios/clean_web_data.csv')
df_experiment = pd.read_csv('Datasets_limpios/clean_experiment_clients.csv')

In [27]:
# unimos los 3 dataframes

# Unimos web data con experiment para saber quién es Control y quién es Test
df = df_web.merge(df_experiment, on='client_id', how='inner')

# Añadimos los datos demográficos
df = df.merge(df_demo, on='client_id', how='inner')

print(df.shape)
print(df.columns.tolist())
df.head()

(317235, 14)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']


,client_id,visitor_id,visit_id,process_step,date_time,variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0


In [28]:
df['variation'].unique()

array(['Test', 'Control'], dtype=object)

In [29]:
# 1. Calcular completation rate (tasa de completar)

# Separamos los grupos de la columna variation
control = df[df['variation'] == 'Control']
test    = df[df['variation'] == 'Test']

# Función para calcular el rate
def completion_rate(grupo):
    total       = grupo['client_id'].nunique() # cuenta cuantos clientes únicos hay en en el grupo
    completaron = grupo[grupo['process_step'] == 'confirm']['client_id'].nunique() # de esos clientes, cuántos llegaron hasta el pason confirm
    return completaron / total # divide los clienten que completaron entre el total del clientes

# Calcular tasas
tasa_control = completion_rate(control)
tasa_test    = completion_rate(test)

print(f"Completion rate Control: {tasa_control:.2%}")
print(f"Completion rate Test:    {tasa_test:.2%}")

Completion rate Control: 65.59%
Completion rate Test:    69.29%


In [30]:
# Guardar los 4 números para el z-test
total_control    = control['client_id'].nunique()
total_test       = test['client_id'].nunique()
completed_control = control[control['process_step'] == 'confirm']['client_id'].nunique()
completed_test    = test[test['process_step'] == 'confirm']['client_id'].nunique()

print(f"Control: {completed_control} completaron de {total_control} totales")
print(f"Test:    {completed_test} completaron de {total_test} totales")

Control: 15434 completaron de 23532 totales
Test:    18687 completaron de 26968 totales


Estos números nos dicen lo siguiente:
- En el grupo Control, el 65,59% de usuarios completo el proceso.
- En el grupo Test, el 69,29% de usuarios completó el proceso (casi 4 puntos más que el otro grupo).

Esto ahora toca demostrarlo, que esta diferencia de puntos no es azar. Para ello, usaremos el two-sided z-test porque vamos a comparar proporciones es decir 65% vs 69% de los grupos respectivamente.

In [31]:
# 2. Hipotesis 1
#H0: tasa_test = tasa_control
# The completion rate for the Test group (new design) is equal to the completion rate for the Control group (old design).
#H1: tasa_test ≠ tasa_control
# The completion rate for the Test group (new design) is not equal to the completion rate for the Control group (old design).

# Los datos que necesita el z-test
counts       = [completed_control, completed_test] # cuantos completaron en cada grupo
total_groups = [total_control, total_test] # cuántos usuarios  había en total en cada grupo

# Two-sided = dos colas = hay diferencia en cualquier dirección? es significativa?
stat, p_value = proportions_ztest(counts, total_groups, alternative='two-sided')
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Rechazamos H0: la diferencia en completion rate es estadísticamente significativa")
else:
    print("No rechazamos H0: no hay evidencia suficiente de diferencia")

p-value: 0.0000
Rechazamos H0: la diferencia en completion rate es estadísticamente significativa


Aplicamos un two-sided  z-test para comparar la completion rate del grupo Control (65.59%) con la del grupo Test (69.29%). El test devuelve un p-value inferior 
a 0.05, por lo que rechazamos la hipótesis nula y concluimos que la diferencia en completion rate entre los dos grupos es estadísticamente significativa e imposible que sea casualidad.

Por tanto, se puede decir que  el nuevo diseñode la web hace que más gente termine el proceso. El resultado nos dice que sí, el grupo que usó el diseño nuevo completó un 69% de las veces, frente al 66% del diseño antiguo. Y esta diferencia no es casualidad, los números lo confirman.

In [ ]:
# 3. Hipotesis 2 
# H0: The completion rate for the Test group (new design) is equal to or less than the completion rate for the Control group (old design) increased by 5%.
# H1: The completion rate for the Test group (new design) is greater than the completion rate for the Control group (old design) increased by 5%.